# Galerie cameroun_map — du CSV à la carte, étape par étape

Ce notebook est la porte d'entrée pédagogique du projet. Deux objectifs :

1. **Utiliser** la librairie — produire des cartes à partir de vos données.
2. **Comprendre** la librairie — savoir où tout se passe pour corriger,
   adapter ou étendre le code vous-même.

**Plan :**

| # | Section | Ce que vous apprendrez |
|---|---|---|
| 1 | Installation et données | Mise en place, données de démonstration |
| 2 | Carte minimale | Le pipeline en 4 lignes |
| 3 | `carte.valider()` | Vérifier la jointure avant de publier |
| 4 | Carte interactive | Folium HTML |
| 5 | Méthodes de classification | Quantile, Jenks, Égale, Std |
| 6 | `focus()` | Zoom mode A |
| 7 | `drill_down()` | Zoom mode B |
| 8 | `adapter_dataset` seul | Nettoyer sans cartographier |
| 9 | Niveau département | Pipeline à grande échelle |
| 10 | Style et personnalisation | Palette, fond, police, DPI |
| 11 | Symboles | Icônes prédéfinies et images importées |
| 12 | Annotations | Callout, highlight, valeur |
| 13 | Architecture | Lire et naviguer le code source |
| 14 | Données problématiques | Détecter et corriger un fichier sale |
| 15 | Déboguer une carte fausse | Identifier une jointure silencieusement erronée |
| 16 | Étendre la librairie | Exercice guidé : ajouter une fonctionnalité |

> Le premier `charger_geo()` télécharge les contours depuis GADM (internet
> requis une seule fois, puis mis en cache dans `data/`).


## 1. Installation et données de démonstration

```bash
pip install -e ".[fuzzy]"
```

Les données ci-dessous remplacent un vrai CSV pour rendre le notebook
reproductible sans fichier externe.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cameroun_map import (
    CarteCameroun, Style, SYMBOLES_PREDEFINIS,
    diagnostiquer_dataset, adapter_dataset, classifier_valeurs,
)

np.random.seed(0)

REGIONS = [
    "Adamaoua", "Centre", "Est", "Extrême-Nord", "Littoral",
    "Nord", "Nord-Ouest", "Ouest", "Sud", "Sud-Ouest",
]

df_regions = pd.DataFrame({
    "region": REGIONS,
    "taux_pauvrete": np.random.uniform(20, 80, len(REGIONS)).round(1),
})
df_regions

## 2. Carte minimale (régions)

Quatre lignes suffisent. `charger_metrique()` diagnostique et adapte le
DataFrame automatiquement — il n'a pas besoin d'être propre.


In [ ]:
carte = CarteCameroun(niveau="regions")
carte.charger_geo()
carte.charger_metrique(df_regions, col_nom="region", col_valeur="taux_pauvrete")
carte.diagnostiquer()

In [ ]:
carte.visualiser(
    titre="Taux de pauvreté par région — Cameroun (données simulées)",
    unite="%",
    methode="jenks",
    schema_couleur="OrRd",
    afficher_labels=True,
)

## 3. Vérifier la jointure avant de publier — `carte.valider()`

`diagnostiquer()` dit *combien* de zones ont une valeur.
`valider()` dit *comment* chaque zone a obtenu sa valeur : correspondance
exacte, fuzzy avec quel score, ou absente.

On introduit volontairement des noms mal orthographiés pour voir ce que
`valider()` détecte.


In [ ]:
df_avec_defauts = pd.DataFrame({
    "region": [
        "Adamaoua", "centre", "Est", "extreme nord", "Littoral",
        "Nord", "Nord Ouest", "ouest", "Sud", "Sud-ouest",
    ],
    "taux_pauvrete": np.random.uniform(20, 80, len(REGIONS)).round(1),
})

carte2 = CarteCameroun(niveau="regions")
carte2.charger_geo()
carte2.charger_metrique(df_avec_defauts, col_nom="region", col_valeur="taux_pauvrete")
table_validation = carte2.valider(seuil_alerte=85)

Deux choses à surveiller :

- **score < seuil_alerte** : la correspondance a réussi mais reste douteuse.
  Vérifiez manuellement que "extreme nord" → "Extrême-Nord" est bien correct.
- **`[join][ALERTE]`** au moment de `charger_metrique()` : deux noms d'origine
  différents ont convergé vers la même zone géographique — les valeurs sont
  moyennées silencieusement, ce qui peut masquer une erreur de saisie.

`table_validation` est un DataFrame exportable : `table_validation.to_csv("validation.csv")`.


## 4. Carte interactive (Folium)

Même jointure, rendu HTML autonome — utile pour explorer avant de figer
une carte statique pour publication.


In [ ]:
carte.visualiser_interactif(
    titre="Taux de pauvreté — Cameroun",
    moteur="folium",
    unite="%",
    sortie="galerie_carte_interactive.html",
)

## 5. Comparer les méthodes de classification

Le choix de la méthode change radicalement la lecture d'une carte sur les
**mêmes données**. Ne publiez jamais une carte choroplèthe sans mentionner
la méthode utilisée dans la légende ou le rapport.


In [ ]:
gdf_joint = carte.gdf_joint.to_crs("EPSG:32632")
methodes = ["quantile", "jenks", "égale", "std"]

fig, axes = plt.subplots(2, 2, figsize=(12, 13))
fig.suptitle("Même donnée, quatre lectures différentes", fontweight="bold")

for ax, methode in zip(axes.flat, methodes):
    from matplotlib.colors import BoundaryNorm
    vals = gdf_joint["valeur"].dropna()
    bornes, etiquettes = classifier_valeurs(vals, methode, 5)
    cmap = plt.get_cmap("YlOrRd", len(bornes) - 1)
    norm = BoundaryNorm(bornes, cmap.N)
    gdf_joint[~gdf_joint["manquant"]].plot(
        ax=ax, column="valeur", cmap=cmap, norm=norm,
        edgecolor="white", linewidth=0.5,
    )
    gdf_joint[gdf_joint["manquant"]].plot(ax=ax, color="#D0D0D0", edgecolor="white")
    ax.set_title(methode, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()

> **À retenir** — `jenks` (ruptures naturelles) révèle les clusters réels
dans les données. `quantile` garantit des classes de même taille.
`égale` est lisible mais peut être trompeuse sur des distributions asymétriques.
`std` est utile quand la distance à la moyenne est le message.


## 6. Zoom sur une zone — `focus()` (mode A)

`focus()` filtre sur une zone sans changer de niveau. L'objet retourné est un
**nouveau** `CarteCameroun` — la carte d'origine reste intacte.


In [ ]:
zoom_littoral = carte.focus("Littoral")
zoom_littoral.visualiser(
    titre="Région Littoral — taux de pauvreté",
    unite="%",
    afficher_contexte=True,   # fond national grisé
)

## 7. Subdivisions internes — `drill_down()` (mode B)

`drill_down()` change de niveau à l'intérieur d'une zone parente.


In [ ]:
df_departements_littoral = pd.DataFrame({
    "departement": ["Wouri", "Moungo", "Nkam", "Sanaga-Maritime"],
    "couverture_vaccinale": np.random.uniform(60, 95, 4).round(1),
})

zoom_dept = carte.drill_down(
    zone="Littoral",
    niveau_interieur="departements",
    df_metrique=df_departements_littoral,
    col_nom_metrique="departement",
    col_valeur_metrique="couverture_vaccinale",
)
zoom_dept.visualiser(
    titre="Départements du Littoral — couverture vaccinale",
    unite="%",
    afficher_contexte=True,
)

## 8. Nettoyer un fichier brut — `adapter_dataset`

`adapter_dataset` est indépendant de la cartographie. Il sert à nettoyer
un fichier même si vous ne voulez pas faire de carte.


In [ ]:
df_brut = pd.DataFrame({
    "region": ["littoral", "centre", "est", "littoral"],  # doublon
    "2022": [12.1, 8.4, 15.0, 12.6],
    "2023": [11.5, 8.0, 14.2, 11.9],
})
diagnostiquer_dataset(df_brut)

In [ ]:
df_propre = adapter_dataset(df_brut)
df_propre

`adapter_dataset` a converti le format large (colonnes 2022/2023) en lignes
par période, et agrégé le doublon `"littoral"` par moyenne. La normalisation
orthographique vers les noms officiels GADM a lieu plus tard, dans
`charger_metrique()`.


## 9. Niveau département

Même pipeline, granularité plus fine (58 zones au lieu de 10).


In [ ]:
carte_dept = CarteCameroun(niveau="departements")
carte_dept.charger_geo()
print(f"{len(carte_dept.gdf)} départements chargés.")
# carte_dept.charger_metrique(df_departements)  ← vos 58 départements ici

## 10. Style et personnalisation graphique

`Style` est un objet séparé, réutilisable sur plusieurs cartes d'un même rapport.
Il contrôle palette, fonds, contours, police, DPI, et zones manquantes.


In [ ]:
from cameroun_map import Style

# ── Style mode publication : fond blanc, palette bleue sobre ─────────────────
style_pub = Style(
    fond_blanc=True,                                     # fond_figure="white", fond_carte="#F0F0F5"
    palette=["#EDF5FF", "#9EC8F0", "#2E86AB", "#1A5276"],  # 4 couleurs hex ordonnées
    taille_titre=14,
    dpi=300,                                             # haute résolution pour export
)

print("fond_figure :", style_pub.fond_figure)
print("fond_carte  :", style_pub.fond_carte)
print("palette     :", style_pub.palette)

In [ ]:
# Même données, même jointure, look radicalement différent
carte_pub = CarteCameroun(niveau="regions", style=style_pub)
carte_pub.charger_geo()
carte_pub.charger_metrique(df_regions, col_nom="region", col_valeur="taux_pauvrete")

carte_pub.visualiser(
    titre="Taux de pauvreté par région",
    sous_titre="Les régions du Nord affichent les niveaux les plus élevés",  # message éditorial
    unite="%",
    methode="jenks",
    source="INS Cameroun, 2024 — données simulées",
    afficher_labels=True,
)

`sous_titre` porte le **message** (la conclusion), pas le sujet.
`source` remplace le texte de crédit automatique en bas à gauche.

> Pour un rapport cohérent : créez un objet `Style` une fois, passez-le
> à chaque `CarteCameroun` — toutes vos cartes auront le même look.


## 11. Couches — symboles

`ajouter_symboles()` superpose des icônes sur la carte de base.
Deux modes de placement, deux types de symbole.

### Catalogue des symboles prédéfinis


In [ ]:
print("Symboles disponibles :")
for nom, code_sym in SYMBOLES_PREDEFINIS.items():
    print(f"  {nom!r:15} → {code_sym}")

In [ ]:
# ── Mode A : placement par nom de zone (centroïde automatique) ──────────────
df_alertes = pd.DataFrame({
    "region":  ["Extrême-Nord", "Nord", "Adamaoua"],
    "niveau":  ["critique",     "attention", "attention"],
})

carte_sym = CarteCameroun(niveau="regions", style=Style(fond_blanc=True))
carte_sym.charger_geo()
carte_sym.charger_metrique(df_regions, col_nom="region", col_valeur="taux_pauvrete")

carte_sym.ajouter_symboles(
    df_alertes,
    col_zone="region",           # placement au centroïde de la zone
    col_categorie="niveau",
    symboles={"critique": "⚠", "attention": "▲"},
    couleurs={"critique": "#E74C3C", "attention": "#F39C12"},
    taille=16,
)

carte_sym.visualiser(
    titre="Zones d'alerte sanitaire — Cameroun",
    sous_titre="3 régions en situation critique ou à surveiller",
    source="Exemple — données simulées",
)

In [ ]:
# ── Mode B : placement par coordonnées lat/lon explicites ───────────────────
df_villes = pd.DataFrame({
    "nom":  ["Douala", "Yaoundé", "Garoua"],
    "lat":  [  4.051,    3.848,    9.301],
    "lon":  [  9.704,   11.516,   13.398],
    "type": ["port",    "capitale", "nord"],
})

carte_villes = CarteCameroun(niveau="regions", style=Style(fond_blanc=True))
carte_villes.charger_geo()
carte_villes.charger_metrique(df_regions, col_nom="region", col_valeur="taux_pauvrete")

carte_villes.ajouter_symboles(
    df_villes,
    col_lat="lat",               # coordonnées explicites WGS84
    col_lon="lon",
    col_categorie="type",
    symboles={"port": "●", "capitale": "★", "nord": "●"},
    couleurs={"port": "#2980B9", "capitale": "#8E44AD", "nord": "#E67E22"},
    taille=14,
)

carte_villes.visualiser(
    titre="Principales villes du Cameroun",
    source="Exemple — données simulées",
    afficher_labels=True,
)

> **Images importées** : remplacez `type="predefined"` par `type="image"` et
> fournissez un dict `symboles={"type": "chemin/vers/icone.png"}`.
> Pillow (`pip install Pillow`) doit être installé.


## 12. Couches — annotations

`ajouter_annotation()` place un texte éditorial sur la carte.
Trois styles : `callout` (flèche), `highlight` (boîte sur place), `valeur` (texte brut).


In [ ]:
carte_ann = CarteCameroun(niveau="regions", style=Style(fond_blanc=True))
carte_ann.charger_geo()
carte_ann.charger_metrique(df_regions, col_nom="region", col_valeur="taux_pauvrete")

# callout : boîte avec flèche pointant vers le centroïde de la zone
carte_ann.ajouter_annotation(
    "Extrême-Nord",
    texte="Zone prioritaire
67% de pauvreté",
    style="callout",
    couleur_fond="#FDECEA",
    offset=(60, 30),    # décalage en km : 60 km à droite, 30 km au-dessus
)

# highlight : boîte directement sur la zone, sans flèche
carte_ann.ajouter_annotation(
    "Littoral",
    texte="Meilleur score",
    style="highlight",
    couleur_fond="#E8F8E8",
)

# valeur : texte brut en gras, placement par coordonnées lat/lon
carte_ann.ajouter_annotation(
    (3.848, 11.516),    # (lat, lon) de Yaoundé
    texte="Yaoundé",
    style="valeur",
    couleur_texte="#8E44AD",
    taille_texte=9,
)

carte_ann.visualiser(
    titre="Annotations multi-styles",
    source="Exemple — données simulées",
)

> **Chaînage** : `ajouter_symboles()` et `ajouter_annotation()` retournent
> `self`, donc vous pouvez écrire :
> ```python
> carte.ajouter_symboles(...).ajouter_annotation(...).visualiser(...)
> ```
> Pour effacer les couches sans recréer l'objet : `carte.reinitialiser_couches()`.


## 13. Architecture — lire et naviguer le code

Avant de modifier la librairie, il faut savoir où tout se passe.
Cette section utilise `inspect` pour afficher le code source réel.

### Les trois fichiers, trois responsabilités


In [ ]:
import inspect
from cameroun_map import (
    adapter_dataset, CarteCameroun, joindre_metrique,
)
import cameroun_map.cameroun_viz as viz
import cameroun_map.adapter_dataset as ads

# Taille de chaque module
for nom, module in [("cameroun_viz", viz), ("adapter_dataset", ads)]:
    lignes = inspect.getsource(module).count("\n")
    fonctions = [n for n, _ in inspect.getmembers(module, inspect.isfunction)
                 if not n.startswith("_")]
    print(f"{nom:25} {lignes:4} lignes  |  fonctions publiques : {fonctions}")

In [ ]:
# Signature complète de charger_metrique — ce que vous pouvez passer
print(inspect.signature(CarteCameroun.charger_metrique))
print()
# Signature de adapter_dataset — le pipeline de nettoyage
print(inspect.signature(adapter_dataset))
print()
# Signature de joindre_metrique — la jointure géographique
print(inspect.signature(joindre_metrique))

In [ ]:
# Voir les 30 premières lignes de normaliser_nom — le fuzzy matching
from cameroun_map import normaliser_nom
src = inspect.getsource(normaliser_nom)
for i, ligne in enumerate(src.splitlines()[:30], 1):
    print(f"{i:3}  {ligne}")

### Carte mentale : "je veux modifier X, j'édite Y"

| Ce que je veux changer | Fichier | Fonction/méthode |
|---|---|---|
| Comment les noms de zones sont corrigés | `adapter_dataset.py` | `normaliser_nom()` |
| Comment les doublons sont agrégés | `adapter_dataset.py` | `agreger_doublons()` |
| Comment les valeurs sont découpées en classes | `cameroun_viz.py` | `classifier_valeurs()` |
| Comment la jointure est faite | `cameroun_viz.py` | `joindre_metrique()` |
| Ce que `visualiser()` dessine | `cameroun_viz.py` | `carte_statique()` |
| Le rendu avec fond grisé (focus/drill_down) | `cameroun_viz.py` | `_carte_avec_contexte()` |
| L'export Power BI | `export_powerbi.py` | `exporter_pour_powerbi()` |

> **Règle de navigation** : si le problème vient des données avant la carte
> → `adapter_dataset.py`. Si le problème vient de la carte elle-même
> → `cameroun_viz.py`, fonctions commençant par `carte_` ou `_carte_`.


## 14. Données réelles problématiques

Un vrai fichier de terrain cumule rarement un seul défaut. Voici un CSV
simulé avec cinq problèmes simultanés : format large, noms en franglais,
doublon, valeur manquante, et ligne de total parasite.


In [ ]:
df_terrain = pd.DataFrame({
    "Zone":   ["LITTORAL", "centre", "Est", "Extreme-Nord",
               "littoral",  "Ouest",  "TOTAL CAMEROUN", None],
    "2021":   [12.1,  8.4, 15.0, 67.2,   12.6, 22.3, 999.0, 5.0],
    "2022":   [11.5,  8.0, 14.2, 68.1,   11.9, 21.8, 998.0, None],
    "Source": ["INS", "INS", "INS", "INS", "terrain", "INS", "INS", "INS"],
})
print("DataFrame brut :")
print(df_terrain.to_string())

In [ ]:
# Étape 1 — diagnostiquer sans modifier
print("=" * 60)
print("DIAGNOSTIC")
print("=" * 60)
diagnostiquer_dataset(df_terrain)

In [ ]:
# Étape 2 — adapter (détecte format large, agrège doublons, gère NaN)
df_nettoye = adapter_dataset(df_terrain, col_zone="Zone")

print("Après adaptation :")
print(df_nettoye.to_string())
print()
print("⚠ La ligne 'TOTAL CAMEROUN' est présente — à filtrer manuellement :")
df_nettoye = df_nettoye[~df_nettoye["zone"].str.upper().str.contains("TOTAL", na=False)]
print(df_nettoye.to_string())

In [ ]:
# Étape 3 — joindre et valider : voir ce que le fuzzy matching a fait
carte_terrain = CarteCameroun(niveau="regions")
carte_terrain.charger_geo()
carte_terrain.charger_metrique(df_nettoye, auto_adapter=False,
                               col_nom="zone", col_valeur="valeur")

validation = carte_terrain.valider(seuil_alerte=80)
print("\nCorrespondances à vérifier manuellement :")
print(validation[validation["methode"].isin(["fuzzy", "non_trouve"])].to_string())

### Leçons de cette section

1. **Ne faites jamais confiance au pipeline sans `valider()`** — un mauvais
   nom peut matcher silencieusement une zone erronée.
2. **Les lignes parasites** (totaux, en-têtes dupliqués) doivent être filtrées
   manuellement *après* `adapter_dataset` — la librairie ne peut pas deviner
   qu'un "TOTAL" n'est pas une zone géographique.
3. **Les valeurs `None`/`NaN`** dans la colonne zone sont ignorées par
   `adapter_dataset` — elles n'apparaîtront pas sur la carte.


## 15. Déboguer une carte fausse

La carte la plus dangereuse n'est pas celle qui plante — c'est celle qui
s'affiche sans erreur mais avec des valeurs erronées. Ce cas arrive quand
la jointure réussit sur les mauvais noms.


In [ ]:
from shapely.geometry import Polygon
import geopandas as gpd

# GeoDataFrame factice — pas besoin de GADM pour cet exercice
gdf_test = gpd.GeoDataFrame({
    "NAME_1": ["Littoral", "Centre", "Est", "Nord", "Adamaoua",
               "Ouest",   "Sud",    "Extrême-Nord", "Nord-Ouest", "Sud-Ouest"],
    "geometry": [Polygon([(i, 0), (i+1, 0), (i+1, 1), (i, 1)])
                 for i in range(10)],
}, crs="EPSG:4326")

# Dataset avec un bug subtil : "Nord" et "Nord-Ouest" sont présents tous les deux
# → le fuzzy matching peut les confondre selon le seuil
df_bug = pd.DataFrame({
    "zone":   ["Littoral", "Nord", "Nord-Ouest", "Centre", "Est"],
    "valeur": [    42.0,    68.0,        25.0,     31.0,   55.0],
})
print("Dataset source :")
print(df_bug)

In [ ]:
from cameroun_map import joindre_metrique

# Jointure directe pour voir ce qui se passe
gdf_joint_test = joindre_metrique(
    gdf_test, df_bug,
    col_geo="NAME_1", col_metrique="zone", col_valeur="valeur",
    fuzzy=True,
)

# Inspecter le résultat
colonnes = ["NAME_1", "valeur", "manquant", "_nom_origine", "_match_score", "_match_methode"]
colonnes_presentes = [c for c in colonnes if c in gdf_joint_test.columns]
print(gdf_joint_test[colonnes_presentes].to_string())

In [ ]:
# Identifier le problème : quelle zone a reçu une valeur d'une autre zone ?
if "_match_score" in gdf_joint_test.columns:
    suspects = gdf_joint_test[
        gdf_joint_test["_match_methode"].isin(["fuzzy"]) &
        (gdf_joint_test["_match_score"] < 90)
    ][colonnes_presentes]
    if suspects.empty:
        print("Aucune correspondance suspecte détectée (seuil 90%).")
    else:
        print("⚠ Correspondances à vérifier :")
        print(suspects.to_string())
else:
    print("Colonnes de validation absentes — appelez charger_metrique() via CarteCameroun.")

### Règle de débogage

1. **La carte est vide ou toutes les zones sont grises** → le nom de la
   colonne zone est mal orthographié, ou `auto_adapter=True` n'a pas
   détecté la bonne colonne. Précisez `col_nom=` explicitement.

2. **Certaines zones sont grises** → `valider()` — vérifiez la colonne
   `_nom_origine` pour voir ce que votre CSV contenait réellement.

3. **Les valeurs semblent fausses** → collision de noms fuzzy.
   Utilisez `valider(seuil_alerte=90)` et inspectez les lignes
   `methode=fuzzy` avec un score proche du seuil.

4. **Crash `BoundaryNorm`** → une seule zone avec données (ex: après
   `focus()`). Déjà corrigé dans `classifier_valeurs()` — si vous voyez
   ça, mettez à jour la librairie.


## 16. Étendre la librairie — exercice guidé

La meilleure façon de comprendre une librairie est d'y ajouter quelque
chose de petit. Voici deux exercices progressifs.

---

### Exercice A — Ajouter un paramètre `afficher_nord`

**Objectif** : permettre d'afficher ou masquer la rose des vents via
`carte.visualiser(afficher_nord=False)`.

**Où modifier** : [src/cameroun_map/cameroun_viz.py](../src/cameroun_map/cameroun_viz.py)

**Étapes** :

1. Dans `carte_statique()`, cherchez les lignes :
   ```python
   ax.annotate("N", xy=(0.97, 0.97), xycoords="axes fraction", ...)
   ax.annotate("↑", xy=(0.97, 0.94), xycoords="axes fraction", ...)
   ```

2. Enveloppez-les dans un `if afficher_nord:`.

3. Ajoutez `afficher_nord: bool = True` aux paramètres de `carte_statique()`.

4. Propagez le paramètre dans `CarteCameroun.visualiser()` :
   ajoutez `afficher_nord=True` à sa signature et passez-le à
   `carte_statique(..., afficher_nord=afficher_nord)`.

5. Vérifiez que `carte.visualiser(afficher_nord=False)` ne plante pas.

---

### Exercice B — Ajouter une méthode `exporter_csv_joint()`

**Objectif** : exporter le GeoDataFrame joint (géométrie + données) en CSV,
sans la colonne géométrie.

**Où ajouter** : dans la classe `CarteCameroun`.

**Implémentation** :
```python
def exporter_csv_joint(self, chemin: str, encoding: str = "utf-8-sig"):
    assert self.gdf_joint is not None, "Appelez d'abord charger_metrique()"
    df = self.gdf_joint.drop(columns="geometry").copy()
    df.to_csv(chemin, index=False, encoding=encoding)
    print(f"[ok] CSV exporté : {chemin}")
    return self
```

L'encodage `utf-8-sig` (avec BOM) garantit que Excel et Power BI lisent
les accents correctement sous Windows.


In [ ]:
# Vérification de l'exercice B — testez votre implémentation ici
# (après avoir ajouté la méthode dans cameroun_viz.py et rechargé le kernel)

# carte.exporter_csv_joint("jointure_regions.csv")
# import pandas as pd; print(pd.read_csv("jointure_regions.csv").head())

print("Colonnes disponibles dans gdf_joint (sans geometry) :")
cols = [c for c in carte.gdf_joint.columns if c != "geometry"]
print(cols)

### Principes pour étendre sans casser

1. **Lisez `CONTEXTE_PROJET.md`** avant toute modification — la section 12
   liste les décisions de conception à ne pas remettre en question.

2. **Testez avec le snippet de non-régression** (section 14 de
   `CONTEXTE_PROJET.md`) après chaque modification.

3. **Ne modifiez jamais `adapter_dataset.py` pour importer depuis
   `cameroun_viz.py`** — cela créerait une dépendance circulaire.

4. **`focus()` et `drill_down()` doivent toujours retourner un nouvel
   objet `CarteCameroun`**, pas modifier `self` — c'est un invariant central.

5. **Si vous êtes bloqué** : copiez le contenu de `CONTEXTE_PROJET.md`
   dans n'importe quelle IA (Claude, ChatGPT, Gemini) et décrivez ce que
   vous voulez faire. Le document contient suffisamment de contexte pour
   obtenir une aide précise sans historique de conversation.


## Pour finir

Ce notebook couvre l'usage, la validation, la personnalisation, l'architecture
et l'extension de la librairie. Ce que vous devez maintenant faire, dans
l'ordre de priorité :

| Priorité | Action | Pourquoi |
|---|---|---|
| **1** | Tester avec un vrai fichier de données | La seule façon de savoir ce qu'`adapter_dataset` rate vraiment |
| **2** | Écrire 3 tests pytest (`normaliser_nom`, `adapter_dataset`, `classifier_valeurs`) | Filet de sécurité avant toute modification |
| **3** | Maintenir `CONTEXTE_PROJET.md` à jour | Votre mémoire du projet pour vous et pour une IA future |
| **4** | Implémenter une feature de la Phase 2 | Le meilleur apprentissage est de casser et réparer soi-même |

**Règle d'or avant de publier une carte :**
`diagnostiquer()` dit que ça a marché. `valider()` dit que c'est juste.
Les deux sont nécessaires — aucun ne remplace l'autre.
